# Hard Negative Mining trên Google Colab (GPU)

Pipeline tương đương `embedding_project/scripts/build_hard_negatives.py` nhưng:
- Embedding chạy trên **GPU T4** (Colab free) — nhanh gấp 5–10× CPU
- Qdrant trỏ thẳng vào cloud server (HTTP) đã deploy sẵn
- Đọc dữ liệu từ **Google Drive** và ghi output về Drive
- Có thể chạy **smoke-test 50 query** trước khi chạy full ~225k query

## Trước khi chạy
1. Đẩy code + notebook này lên GitHub repo (xem script bash cuối cell)
2. Tạo thư mục `/content/drive/MyDrive/DATN/` trên Drive của bạn
3. Upload 2 file:
   - `Dataset - DATN.csv` (corpus)
   - `llm_queries_all.jsonl` (queries)
4. Chạy notebook từ trên xuống dưới, điền API key khi được hỏi

## 1. Mount Google Drive

Chứa CSV corpus + queries JSONL + nơi ghi output.

In [ ]:
from google.colab import drivedrive.mount('/content/drive')import osDATA_DIR = '/content/drive/MyDrive/DATN'DATA_DATA = f'{DATA_DIR}/data'os.makedirs(DATA_DATA, exist_ok=True)print('Đã mount Drive. Files:')!ls -la "{DATA_DATA}" | head -20

## 2. Cài đặt dependencies

In [ ]:
%pip install -q -U \    sentence-transformers \    qdrant-client \    openai \    python-dotenv \    pandas \    tqdm \    torch  --index-url https://download.pytorch.org/whl/cu121import torchprint('GPU available:', torch.cuda.is_available())print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. Lấy code từ GitHub

Tùy chọn: nếu chưa push lên GitHub thì dùng cell dưới (manual upload).

In [ ]:
# Clone repo (không bao gồm data lớn trong repo)# TODO: thay YOUR_USER/REPO bằng repo thật của bạnGITHUB_REPO = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"%cd /content!git clone --depth 1 "$GITHUB_REPO" repo 2>&1 | tail -3%cd /content/repo/embedding_project/notebooks!ls -la

In [ ]:
import syssys.path.insert(0, '/content/repo/embedding_project/notebooks')import hardneg_e5base_colabprint('Module loaded:', hardneg_e5base_colab.__file__)

## 4. Cấu hình API key

Dùng `getpass` để tránh lộ key trong cell output.

In [ ]:
import os, getpassos.environ['NVIDIA_API_KEY_1'] = getpass.getpass('NVIDIA_API_KEY_1 (Maverick 17B): ')os.environ['NVIDIA_API_KEY_2'] = getpass.getpass('NVIDIA_API_KEY_2 (Nemotron 49B): ')os.environ['QDRANT_URL']       = 'http://qdrant.datn-nextgen-suggest.site'os.environ['QDRANT_API_KEY']   = getpass.getpass('QDRANT_API_KEY: ') or 'datn-nextgen-suggest@'print('Đã set keys:',      'NV1=' + os.environ['NVIDIA_API_KEY_1'][:12] + '...',      'NV2=' + os.environ['NVIDIA_API_KEY_2'][:12] + '...',      'QDRANT=' + os.environ['QDRANT_URL'])

## 5. Smoke-test (50 query đầu) — chạy thử trước

Ước lượng ~5–10 phút cho 50 query → xác nhận pipeline ổn trước khi chạy full.

In [ ]:
# Smoke-test: chỉ 50 query đầuhardneg_e5base_colab.main(limit=50)

## 6. Kiểm tra output smoke-test

In [ ]:
import jsonfrom pathlib import Pathout = Path('/content/drive/MyDrive/DATN/data/training_data.jsonl')recs = [json.loads(l) for l in out.open() if l.strip()]print(f'Đã ghi {len(recs)} records')print('\n--- 3 record đầu ---')for r in recs[:3]:    print(f"query: {r['query']}")    print(f"  pos: {r['pos'][0][:100]}...")    print(f"  n_hard={r['n_hard']} | n_easy={r['n_easy']}")    print(f"  neg[0]: {r['neg'][0][:100] if r['neg'] else '(none)'}...")    print()

## 7. Reset output trước khi chạy full

Nếu smoke-test OK, xóa file để chạy lại từ đầu. **BỎ QUA nếu muốn resume từ smoke-test.**

In [ ]:
# CẢNH BÁO: xóa output. Để resume thì KHÔNG chạy cell này.import osout = '/content/drive/MyDrive/DATN/data/training_data.jsonl'if os.path.exists(out):    os.remove(out)    print(f'Đã xóa {out} — pipeline sẽ chạy lại từ đầu.')else:    print('File chưa tồn tại — OK.')

## 8. Chạy FULL — toàn bộ ~225k query

- Ước tính thời gian: ~4–8 giờ (T4 GPU embed rất nhanh, bottleneck là 2×40 RPM LLM judge).
- Nếu Colab disconnect giữa chừng: chạy lại cell này, sẽ auto-resume nhờ `done_set` skip các query đã xong.

In [ ]:
hardneg_e5base_colab.main()  # không truyền limit → chạy full

## 9. Thống kê cuối

In [ ]:
import jsonfrom collections import Counterfrom pathlib import Pathout = Path('/content/drive/MyDrive/DATN/data/training_data.jsonl')recs = [json.loads(l) for l in out.open() if l.strip()]nh = Counter(r['n_hard'] for r in recs)qt = Counter(r.get('query_type','specific') for r in recs)print(f'Total: {len(recs)} records')print(f'Query types: {dict(qt)}')print(f'n_hard distribution: {dict(sorted(nh.items()))}')print(f'avg neg per query: {sum(len(r["neg"]) for r in recs) / max(1,len(recs)):.2f}')print(f'\nFile size: {out.stat().st_size / 1024 / 1024:.1f} MB')print('File path:', out)

## 10. Lấy file về máy

Sau khi chạy xong, file `training_data.jsonl` đã nằm trên Google Drive.
Bạn có thể:
- Mở Drive web và tải về, hoặc
- Dùng cell dưới để copy vào repo Git (nếu file size <100MB) hoặc giữ nguyên trên Drive.

In [ ]:
# Copy file từ Drive về local repo path (Colab workdir)import shutil, ossrc = '/content/drive/MyDrive/DATN/data/training_data.jsonl'dst_dir = '/content/repo/embedding_project/data'os.makedirs(dst_dir, exist_ok=True)shutil.copy(src, dst_dir)print(f'Copied to {dst_dir}/training_data.jsonl')!ls -lh "{dst_dir}/training_data.jsonl"